# M03 — Transformación

[← Anterior](../M02-ingesta-preparacion/04-lab-calidad-limpieza.ipynb) · [Siguiente →](02-lab-enriquecimiento.ipynb)

La regla de negocio es una **columna**, no un `for`. Demo con 4 líneas en memoria (no hace falta el staging).

Ejecuta las celdas **aquí**, en este mismo fichero. No lo copies a otro sitio.

Kernel: **Python (NovaShop)**.


## Arranque

Ejecuta estas dos celdas. Localizan el repo y dejan una `SparkSession` lista.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark('novashop-clase-m03')
print(spark.version, spark.sparkContext.master)


## `withColumn` y GMV

Si el descuento crudo es `1.50`, el GMV **sale negativo**. No es un bug de Spark.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, when, lower, least, lit

lineas = spark.createDataFrame([
    Row(order_id="O1", qty=2, unit_price=10.0, discount=0.10, status="paid", channel="WEB"),
    Row(order_id="O2", qty=1, unit_price=80.0, discount=1.50, status="cancelled", channel="marketplace"),
    Row(order_id="O3", qty=3, unit_price=5.0, discount=0.0, status="paid", channel="app"),
    Row(order_id="O4", qty=1, unit_price=20.0, discount=0.0, status="pending", channel="store"),
])
crudo = lineas.withColumn("gmv_line", col("qty") * col("unit_price") * (1 - col("discount")))
crudo.select("order_id", "discount", "gmv_line").show()


En el lab capas el descuento a 1 y **recalculas** el GMV.


In [ ]:
fact = (
    crudo.withColumn("discount", least(col("discount"), lit(1.0)))
    .withColumn(
        "channel_norm",
        when(lower(col("channel")).isin("web", "app", "store"), lower(col("channel"))).otherwise(lit("other")),
    )
    .withColumn("is_billable", col("status") == "paid")
    .withColumn("gmv_line", col("qty") * col("unit_price") * (1 - col("discount")))
)
fact.select("order_id", "channel", "channel_norm", "discount", "gmv_line", "is_billable").show()


No uses `collect()` / `toPandas()` del fact entero. Si miras, `limit(20).toPandas()`.

**Siguiente:** [lab de enriquecimiento](02-lab-enriquecimiento.ipynb).
